# ArmorVault — PaddleOCR-VL + MiniCPM-V GPU Lab

This notebook runs a non-sensitive public sample through the complete feasibility pipeline. Select **Runtime → Change runtime type → T4 GPU**, then choose **Runtime → Run all**.

The first run downloads several gigabytes of model files and may take 10–20 minutes. Do not upload real identity documents to this notebook.

In [ ]:
import os, subprocess, sys
try:
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
except Exception as exc:
    raise RuntimeError('No NVIDIA GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, then reconnect.') from exc
print('GPU:', gpu)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs('/content/armorvault-lab', exist_ok=True)

## 1. Install GPU runtimes

If Colab asks for a runtime restart after installation, restart it and run all cells again.

In [ ]:
%pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q 'paddleocr[doc-parser]>=3.7.0,<4' 'transformers>=4.51,<5' 'accelerate>=1.4' 'autoawq>=0.2.9' pillow
print('Dependencies installed.')

## 2. Download PaddleOCR's public demo document

This is an official public sample, not a user document.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image

sample_url = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/paddleocr_vl_demo.png'
image_path = Path('/content/armorvault-lab/public_demo.png')
urlretrieve(sample_url, image_path)
display(Image.open(image_path))
print('Public sample:', image_path)

## 3. PaddleOCR-VL extraction

The result is saved locally in the temporary Colab runtime.

In [ ]:
import json, time
from paddleocr import PaddleOCRVL

paddle_dir = Path('/content/armorvault-lab/paddle')
paddle_dir.mkdir(exist_ok=True)
started = time.perf_counter()
pipeline = PaddleOCRVL(
    pipeline_version='v1.6',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
)
init_seconds = time.perf_counter() - started
started = time.perf_counter()
count = 0
for result in pipeline.predict(str(image_path)):
    count += 1
    result.save_to_json(save_path=str(paddle_dir))
    result.save_to_markdown(save_path=str(paddle_dir))
paddle_seconds = time.perf_counter() - started
print({'results': count, 'initializationSeconds': round(init_seconds, 2), 'inferenceSeconds': round(paddle_seconds, 2)})
print('Files:', sorted(p.name for p in paddle_dir.iterdir()))

## 4. Release Paddle GPU memory

MiniCPM is loaded only after PaddleOCR-VL is removed.

In [ ]:
import gc
del pipeline
gc.collect()
try:
    import paddle
    paddle.device.cuda.empty_cache()
except Exception:
    pass
print('Paddle pipeline released.')

## 5. MiniCPM-V 4.5 AWQ understanding

MiniCPM receives the image and PaddleOCR-VL output, then returns structured JSON. The model repository uses trusted remote code; this is suitable for the isolated feasibility runtime, not production without pinning and reviewing a revision.

In [ ]:
import re, torch
from transformers import AutoModel, AutoTokenizer

extraction_parts = []
for path in sorted(paddle_dir.iterdir()):
    if path.suffix.lower() in {'.json', '.md'}:
        extraction_parts.append(path.read_text(encoding='utf-8', errors='replace'))
extraction = '\n\n'.join(extraction_parts)[:60000]
if not extraction:
    raise RuntimeError('PaddleOCR-VL did not produce JSON or Markdown output.')

model_name = 'openbmb/MiniCPM-V-4_5-AWQ'
started = time.perf_counter()
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map='auto',
    low_cpu_mem_usage=True,
).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
minicpm_init_seconds = time.perf_counter() - started

prompt = '''You are a document-understanding layer for Arabic and English documents.
Use the image and PaddleOCR-VL extraction below. Never invent an unreadable character.
Return JSON only with these exact keys:
documentType, documentNumber, holderName, issueDate, expiryDate,
totalAmount, currency, mrzLines.
Use null for uncertain scalar values and [] for absent MRZ lines.
Dates must use YYYY-MM-DD. Do not add explanations.

PaddleOCR-VL extraction:
''' + extraction

started = time.perf_counter()
answer = model.chat(
    msgs=[{'role': 'user', 'content': [Image.open(image_path).convert('RGB'), prompt]}],
    tokenizer=tokenizer,
    enable_thinking=False,
)
minicpm_seconds = time.perf_counter() - started

cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer.strip(), flags=re.I | re.S)
try:
    fields = json.loads(cleaned)
except json.JSONDecodeError:
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise RuntimeError('MiniCPM did not return a JSON object.')
    fields = json.loads(cleaned[start:end + 1])

result = {
    'models': {'extractor': 'PaddleOCR-VL-0.9B', 'understanding': model_name},
    'timingSeconds': {
        'paddleInitialization': round(init_seconds, 2),
        'paddleInference': round(paddle_seconds, 2),
        'miniCPMInitialization': round(minicpm_init_seconds, 2),
        'miniCPMInference': round(minicpm_seconds, 2),
    },
    'fields': fields,
}
result_path = Path('/content/armorvault-lab/result.json')
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))

## 6. Download the result

The downloaded JSON contains only the public demo extraction in this default run.

In [ ]:
from google.colab import files
files.download('/content/armorvault-lab/result.json')